# Config-Driven Forge Layer Transformations

## 🎯 Architecture
This implementation uses **forge_config.py** for metadata-driven, scalable transformations:

* ✅ **15 tables** transformed with **one reusable function**
* ✅ **Explicit print statements** to verify success and row counts
* ✅ **Dependency-aware execution** order (Batch 1 → 2 → 3 → 4)
* ✅ **Optimized with partitioning, caching, and z-ordering**
* ✅ **Data quality validation** built-in

## 📋 What Gets Applied:
* Timestamp fixes (STRING → TIMESTAMP)
* String trimming/uppercase/lowercase
* Calculated columns (trip_duration, experience_years, health_scores, etc.)
* Risk categorizations and flags
* Partitioning by business keys
* Metadata tracking (forge_timestamp)

## 🔄 Execution Order:
1. **Batch 1**: Master tables (gps, route_master, vehicle_master, driver_master, trip_master)
2. **Batch 2**: Telemetry (core_engine, core_telemetry, driver_behavior)
3. **Batch 3**: Operational (fuel_transactions, insurance_claims, maintenance, weather)
4. **Batch 4**: Documents (accident_reports_batch4, insurance_claims_batch4, metadata_batch4_files)

In [0]:
# Import required libraries
import sys
import importlib
from pyspark.sql.functions import (
    col, expr, trim, upper, lower, current_timestamp, to_timestamp,
    round as spark_round, when, lit
)
from pyspark.sql import DataFrame
from datetime import datetime

# Add metadata path and import forge configuration
sys.path.append('/Workspace/Repos/akshaymanikuttan05@gmail.com/AxioGo/axiogo_lakehouse/metadata')

# Force reload to pick up config changes
import forge_config
importlib.reload(forge_config)
from forge_config import (
    FORGE_TABLES,
    TRANSFORMATION_ORDER,
    DATA_QUALITY_CHECKS,
    OPTIMIZATION_CONFIG,
    FORGE_SUMMARY,
    get_table_config
)

print("=" * 80)
print("FORGE LAYER CONFIGURATION LOADED")
print("=" * 80)
print(f"Schema: {FORGE_SUMMARY['schema_name']}")
print(f"Total Tables: {FORGE_SUMMARY['total_tables']}")
print(f"Transformation Order: {', '.join(TRANSFORMATION_ORDER[:5])}...")
print("✅ Ready to transform!")
print("=" * 80)

In [0]:
def transform_table(table_name: str) -> dict:
    """
    Transform a table using configuration from forge_config.py
    
    Returns:
        dict: Status info with success flag, row_count, and duration
    """
    try:
        start_time = datetime.now()
        config = get_table_config(table_name)
        
        if not config:
            return {"success": False, "error": f"Config not found for {table_name}"}
        
        # Read source table
        df = spark.table(config['source_table'])
        source_count = df.count()
        
        # Apply timestamp fixes (use PySpark to_timestamp for automatic parsing)
        for ts_col in config.get('timestamp_columns', []):
            if ts_col in df.columns:
                # to_timestamp() handles ISO format automatically in PySpark
                df = df.withColumn(ts_col, to_timestamp(col(ts_col)))
        
        # Trim string columns
        for trim_col in config.get('trim_columns', []):
            if trim_col in df.columns:
                df = df.withColumn(trim_col, trim(col(trim_col)))
        
        # Apply uppercase transformations
        for upper_col in config.get('uppercase_columns', []):
            if upper_col in df.columns:
                df = df.withColumn(upper_col, upper(col(upper_col)))
        
        # Apply lowercase transformations
        for lower_col in config.get('lowercase_columns', []):
            if lower_col in df.columns:
                df = df.withColumn(lower_col, lower(col(lower_col)))
        
        # Add calculated columns using SQL expressions
        for calc_col, sql_expr in config.get('calculated_columns', {}).items():
            df = df.withColumn(calc_col, expr(sql_expr))
        
        # Add forge metadata
        df = df.withColumn('forge_timestamp', current_timestamp())
        df = df.withColumn('source_batch', lit(config.get('batch', 'Unknown')))
        
        # Write with partitioning if specified
        writer = df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
        
        if config.get('partitioning'):
            writer = writer.partitionBy(config['partitioning'])
        
        writer.saveAsTable(config['target_table'])
        
        # Verify write
        target_count = spark.table(config['target_table']).count()
        duration = (datetime.now() - start_time).total_seconds()
        
        return {
            "success": True,
            "source_count": source_count,
            "target_count": target_count,
            "duration_seconds": round(duration, 2),
            "partitioning": config.get('partitioning', 'None'),
            "calculated_columns": len(config.get('calculated_columns', {}))
        }
        
    except Exception as e:
        return {
            "success": False,
            "error": str(e)
        }

print("✅ transform_table() function defined")
print("   - Applies timestamp fixes, string operations, calculated columns")
print("   - Handles partitioning and metadata")
print("   - Returns verification info")

In [0]:
# Note: Caching disabled for serverless compute
print("\n" + "=" * 80)
print("STARTING FORGE TRANSFORMATIONS")
print("=" * 80)

# Track overall stats
total_tables = len(TRANSFORMATION_ORDER)
success_count = 0
failed_tables = []
total_rows_processed = 0
total_duration = 0

# Transform each table in dependency order
for i, table_name in enumerate(TRANSFORMATION_ORDER, 1):
    config = get_table_config(table_name)
    
    print(f"\n[{i}/{total_tables}] Processing: {table_name}")
    print(f"    Batch: {config.get('batch', 'Unknown')} | Priority: {config.get('priority', 'N/A')}")
    print(f"    Description: {config.get('description', 'No description')}")
    print(f"    Source: {config.get('source_table')}")
    print(f"    Target: {config.get('target_table')}")
    
    # Transform the table
    result = transform_table(table_name)
    
    if result['success']:
        print(f"    ✅ SUCCESS")
        print(f"       Rows: {result['source_count']:,} → {result['target_count']:,}")
        print(f"       Duration: {result['duration_seconds']}s")
        print(f"       Calculated Columns: {result['calculated_columns']}")
        print(f"       Partitioning: {result['partitioning']}")
        
        success_count += 1
        total_rows_processed += result['target_count']
        total_duration += result['duration_seconds']
    else:
        print(f"    ❌ FAILED: {result.get('error', 'Unknown error')}")
        failed_tables.append(table_name)

print("\n" + "=" * 80)
print("FORGE TRANSFORMATION SUMMARY")
print("=" * 80)
print(f"Total Tables: {total_tables}")
print(f"✅ Successful: {success_count}")
print(f"❌ Failed: {len(failed_tables)}")
if failed_tables:
    print(f"   Failed tables: {', '.join(failed_tables)}")
print(f"\n📊 Total Rows Processed: {total_rows_processed:,}")
print(f"⏱️  Total Duration: {round(total_duration, 2)}s")
print(f"⚡ Avg Time per Table: {round(total_duration / total_tables, 2)}s")
print("\n✅ All transformations complete!")
print("=" * 80)

In [0]:
# Verify all forge tables were created
print("\n" + "=" * 80)
print("VERIFICATION: FORGE TABLES")
print("=" * 80)

for table_name in TRANSFORMATION_ORDER:
    config = get_table_config(table_name)
    target_table = config['target_table']
    
    if spark.catalog.tableExists(target_table):
        count = spark.table(target_table).count()
        print(f"✅ {table_name:30s} | {count:,} rows")
    else:
        print(f"❌ {table_name:30s} | TABLE NOT FOUND")

print("=" * 80)

In [0]:
# Run sample data quality checks
print("\n" + "=" * 80)
print("DATA QUALITY VALIDATION (SAMPLE)")
print("=" * 80)

for table_name, checks in DATA_QUALITY_CHECKS.items():
    print(f"\n📋 {table_name}")
    target_table = f"workspace.forge.{table_name}"
    
    if not spark.catalog.tableExists(target_table):
        print(f"   ⚠️  Table not found, skipping checks")
        continue
    
    df = spark.table(target_table)
    total_rows = df.count()
    
    # Check nulls in critical columns
    print(f"   Primary Key: {checks['primary_key']}")
    print(f"   Critical Columns ({len(checks['critical_columns'])}):{', '.join(checks['critical_columns'])}")
    
    for col_name in checks['critical_columns']:
        if col_name in df.columns:
            null_count = df.filter(col(col_name).isNull()).count()
            null_pct = (null_count / total_rows * 100) if total_rows > 0 else 0
            
            if null_pct > checks['null_tolerance']:
                print(f"   ❌ {col_name}: {null_pct:.2f}% nulls (threshold: {checks['null_tolerance']}%)")
            else:
                print(f"   ✅ {col_name}: {null_pct:.2f}% nulls")
    
    # Run validation rules
    for rule in checks.get('rules', []):
        try:
            invalid_count = df.filter(~expr(rule['check'])).count()
            if invalid_count > 0:
                print(f"   ⚠️  Rule '{rule['check']}': {invalid_count} violations ({rule['severity']})")
            else:
                print(f"   ✅ Rule '{rule['check']}': All rows valid")
        except Exception as e:
            print(f"   ⚠️  Rule '{rule['check']}': Check failed - {str(e)}")

print("\n" + "=" * 80)
print("✅ Data quality validation complete!")
print("=" * 80)

# Forge Layer (Silver) - Cleaning & Logical Transformations

## Overview
This notebook performs **data quality cleaning and logical transformations** on intake (bronze) tables.

**Silver Layer Philosophy:**
* Clean and standardize data (types, formats, nulls)
* Add logical calculations and enrichments
* Validate and flag quality issues
* Keep source table structure (no dimensional modeling yet)
* **Dimensional modeling (dim/fact) happens in Gold layer**

## Data Inventory from Intake Layer

### Master Data (Batch 1)
* `workspace.intake.driver_master` - Driver profiles and assignments
* `workspace.intake.vehicle_master` - Vehicle specifications
* `workspace.intake.route_master` - Route definitions
* `workspace.intake.trip_master` - Trip transactions

### Telemetry & IoT (Batch 2)
* `workspace.intake.core_telemetry` - 36 vehicle sensor metrics
* `workspace.intake.core_engine` - Engine-specific metrics
* `workspace.intake.gps` - GPS tracking data
* `workspace.intake.driver_behavior` - Behavioral analytics

### Operational Data (Batch 3)
* `workspace.intake.fuel_transactions` - Fuel purchase records
* `workspace.intake.insurance_claims` - Claim submissions
* `workspace.intake.maintenance` - Service records
* `workspace.intake.weather` - Weather conditions

### Document Data (Batch 4)
* `workspace.intake.accident_reports_batch4` - PDF reports
* `workspace.intake.insurance_claims_batch4` - PDF claims
* `workspace.intake.metadata_batch4_files` - Document metadata

## Silver Layer Transformations (Cleaning + Logical)

### 1. Data Type Standardization
* Fix GPS timestamp (STRING → TIMESTAMP)
* Ensure numeric columns are proper types (INT, DOUBLE, BIGINT)
* Standardize date formats across all tables
* Cast boolean flags properly

### 2. Data Quality & Validation
* **Remove duplicates** based on business keys
* **Handle nulls**: Set defaults or flag for business review
* **Validate ranges**: Speed limits, temperatures, fuel levels
* **Add quality flags**: `is_valid_record`, `has_data_issues`
* **Trim whitespace** from string columns

### 3. Logical Calculations (Derived Columns)
* **Trip duration** from start/end timestamps
* **Speeding violations** flag (speed > speed_limit)
* **Fuel consumption** calculations (fuel_level changes)
* **Idle time percentage** from telemetry
* **Distance validation** (GPS vs reported distance)

### 4. Data Enrichment (Preserve Source Schema)
* Add calculated fields to existing tables (no new tables)
* Add metadata: `processing_timestamp`, `source_batch`
* Flatten nested JSON if needed (keep column structure simple)
* Standardize column naming (snake_case)

### 5. Data Completeness
* Flag incomplete records (missing critical fields)
* Add record counts and checksums for reconciliation
* Track data lineage metadata

## Proposed Silver Layer Tables

**1:1 mapping from Intake → Forge** (same structure, cleaned data)

### Master Data Tables (Batch 1)
1. **forge.driver_master** - Cleaned driver profiles + calculated risk flags
2. **forge.vehicle_master** - Cleaned vehicle specs + age/usage metrics
3. **forge.route_master** - Cleaned route definitions + distance validation
4. **forge.trip_master** - Cleaned trips + trip_duration, trip_date, validation flags

### Telemetry Tables (Batch 2)
5. **forge.core_telemetry** - Cleaned sensor data + anomaly flags, derived metrics
6. **forge.core_engine** - Cleaned engine metrics + performance indicators
7. **forge.gps** - Fixed timestamp, speeding flags, location validation
8. **forge.driver_behavior** - Cleaned behavior data + risk scoring

### Operational Tables (Batch 3)
9. **forge.fuel_transactions** - Cleaned fuel data + consumption calculations
10. **forge.insurance_claims** - Cleaned claims + amount validation
11. **forge.maintenance** - Cleaned maintenance + cost/frequency metrics
12. **forge.weather** - Cleaned weather + condition categorization

### Document Tables (Batch 4)
13. **forge.accident_reports_batch4** - PDF metadata + extraction status
14. **forge.insurance_claims_batch4** - PDF metadata + extraction status
15. **forge.metadata_batch4_files** - Cleaned file metadata

**Note:** Dimensional modeling (star schema) will happen in the Gold layer

## Implementation Examples

### Priority 1: Critical Data Quality Fixes
Start with these high-impact transformations: